# 05_dataset_compatibility
Phase 1 보고서와 원자료 기반 감사 결과. 전체 재계산: 프로젝트 루트에서 `./run_phase1.ps1`.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd()
if not (ROOT / 'reports').exists(): ROOT = ROOT.parent
display(Markdown((ROOT / 'reports/markdown/05_compatibility_report.md').read_text(encoding='utf-8')))
display(pd.read_csv(ROOT / 'reports/tables/compatibility.csv').head(60))


# 두 데이터셋의 호환성

| 비교 항목 | GitHub A | Harvard B | 판정 |
| --- | --- | --- | --- |
| 변수 개수 | 41 measured | 41 measured + 11 manipulated | 불일치 |
| 변수 순서 | Time, xmv-1~41 (실제 simout) | 3 metadata, xmeas_1~41, xmv_1~11 | 부분 대응; 헤더 수정 근거 필요 |
| 변수 정의 | 로컬 소스의 측정 정의 | 원래 변수명 유지; 상세 단위표 없음 | 공통 계열이나 동일 조건 미확정 |
| 샘플링 주기 | 1분 (실측) | 3분 (공식) | 불일치 |
| Fault ID | 1~21, 21 충돌 | 1~20 | 21 제외; 16~20 원인 UNKNOWN |
| 운전 모드 | 1,3,4; 고장은 1만 | 공식 API에 명시 없음 | 미확인 |
| 주입 시각 | 모델 명목 10h; 파일별 미확인 | Training 1h / Testing 8h (공식) | 불일치 |
| 시뮬레이터 조건 | 개정 C, Simulink; batch seed rand() | 시드 분할 설명, 제어기/빌드 미기재 | 동일성 미확인 |
| 독립성 | 중복 시계열 존재, seed 로그 없음 | Train/Test seeds non-overlap (공식) | A-B 독립성 확증 불가 |

전체 측정값의 동일 길이·동일 순서·float64 정확 일치 시계열 비교에서 A-B 일치 0건이다. A를 3분 간격으로 phase 0/1/2분에서 추출하고 선두 500/960개를 B와 비교한 1224개 후보 중 전체 일치 0건, 선두 20개 일치 0건이다. 비교는 후보 xmeas 대응에만 사용했으며 두 데이터를 합친 학습 테이블을 만들지 않았다. 수치 정밀도/시작 시각/서로 다른 제어기 때문에 불일치는 독립 실행의 증명이 아니다. 모든 sliding window, 근사 일치, 동적 정합을 검사한 것은 아니다.

B는 공식 설명에 따라 Training 20개 샘플/Testing 160개 샘플까지 고장 주입 전이다(3분 간격). 이후를 명목 고장 구간으로 삼을 수 있으나 주입 경계의 정확한 기록 규약은 별도 확인해야 한다. faultNumber가 파일 전체에 부여되어도 주입 전 센서를 실제 고장 상태로 오인하면 안 된다. sample*3분 시간축은 공식 25/48h 길이와 일치하는 해석이며 첫 센서 기록이 t=0인지 3분인지에 대한 원 생성 로그는 없다.

A는 공정 이해와 가설 수립에 사용하고 B를 별도의 모델 개발 데이터로 사용하는 것은 가능하다. 두 자료의 행을 직접 합치거나 A 고장 반응을 B 원인 정답으로 사용하는 것은 현재 근거로 허용되지 않는다. B의 Test 기술통계를 탐색했으므로 향후 모델 선택에는 사용하지 않고 별도 train 실행 holdout 또는 새로운 검증 실행으로 분석 선택의 영향을 관리해야 한다.

A 정상 50h 전체와 B 정상 Training run 1의 공통 이름 후보에 대한 평균/표준편차 차이는 candidate_normal_distribution_comparison.csv에 기록했다. 기간·주기·제어 조건이 다른 기술통계이며 동일 분포 가설 검정이나 단위 동일성 검증을 대체하지 않는다. Mode 1/3/4 정상 통계는 a_normal_mode_statistics.csv에 분리했다.


## 출처와 검증 범위

- Dataset A: https://github.com/Blound04/tennessee-eastman-dataset (MIT). 로컬 README, simulator/auto_run.m, Mode_1_Init.m, MultiLoop_mode1.mdl, TEplot.m, temexd_mod.c를 대조했다. 로컬 checkout의 commit은 미확인이다. 다운로드한 Excel은 원래 LFS 포인터의 SHA256과 일치한다.
- Dataset B: https://doi.org/10.7910/DVN/6C3JR1 . 공식 API의 V1.0 메타데이터를 reports/sources/harvard_metadata.json에 보존했다. 배포 설명은 public-domain dedication과 면책 조건을 포함한다. 로컬 파일의 공식 MD5 일치 여부는 감사 표에 기록했다.
- 수치 결과는 로컬 원본의 재현 가능한 기술통계다. Phase 1에서 모델을 학습하거나 Test로 모델을 선택하지 않았다. 자료는 화학공정 시뮬레이션이며 실제 반도체 공정 검증 결과가 아니다.


,비교 항목,GitHub A,Harvard B,판정
0,변수 개수,41 measured,41 measured + 11 manipulated,불일치
1,변수 순서,"Time, xmv-1~41 (실제 simout)","3 metadata, xmeas_1~41, xmv_1~11",부분 대응; 헤더 수정 근거 필요
2,변수 정의,로컬 소스의 측정 정의,원래 변수명 유지; 상세 단위표 없음,공통 계열이나 동일 조건 미확정
3,샘플링 주기,1분 (실측),3분 (공식),불일치
4,Fault ID,"1~21, 21 충돌",1~20,21 제외; 16~20 원인 UNKNOWN
5,운전 모드,"1,3,4; 고장은 1만",공식 API에 명시 없음,미확인
6,주입 시각,모델 명목 10h; 파일별 미확인,Training 1h / Testing 8h (공식),불일치
7,시뮬레이터 조건,"개정 C, Simulink; batch seed rand()","시드 분할 설명, 제어기/빌드 미기재",동일성 미확인
8,독립성,"중복 시계열 존재, seed 로그 없음",Train/Test seeds non-overlap (공식),A-B 독립성 확증 불가
